# Fine-Tuning

ファインチューニングでは、、基本的には通常のPyTorchの流れに従い学習を行います。

- データセットの読込と前処理（主にTokenizerとcollator）
- 学習
- 性能評価

以下の例では、2シーケンスのバッチを1つ用いて、分類モデルを1epochファインチューニングしています（データ量もepoch数も圧倒的に足りないので、もちろん性能は出ません）。

In [ ]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Same as before
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# This is new
batch["labels"] = torch.tensor([1, 1])

optimizer = AdamW(model.parameters())
loss = model(**batch).loss
loss.backward()
optimizer.step()

より実践的な実データを用いてファインチューニングを行う流れを解説していきます

## データセットの読込と前処理

Transformersでは、LLM向けのデータセットを多く含む`datasets`ライブラリを提供しています。
ここで提供されているデータセットは、学習だけでなくベンチマークとしての利用も想定されているものが多いです。

使用できるデータセットは[こちらのHuggingFaceカタログ](https://huggingface.co/datasets)から検索できます。

datasetsライブラリでは、`load_datasets`という関数で、第一引数にデータセット名（ベンチマーク名）を、第二引数にサブデータセット名を渡すことでデータセットを読み込めます。以下の例では、[GLUE](https://gluebenchmark.com/)という英語向けベンチマークに含まれる、[MRPC](https://aclanthology.org/I05-5002.pdf)というデータセットを読み込んでいます。このデータセットは1サンプル（row）あたり2個の文からなり、両方の文が同じ意味であるかを示すラベルが付与されています。すなわち、言い換えタスク（paraphrases）を行うためのデータセットです。

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

データセットはサブセット（train, validation, test）がまとめられたDatasetDict形式で取得され、個々のサブセットは`datasets.Dataset`というクラスで保持されます。

このクラスはPyTorchの`torch.utils.data.Dataset`とは異なり、サイズが大きすぎてメモリに載りきらないというLLM向けデータの特性に合わせ、Apache Arrowというデータフォーマットを使ってハードディスク（SSD）上のデータとメモリをリンクさせ、今まさに処理しようとしているミニバッチの数行分だけを必要な瞬間に高速でディスクから読み出す仕組みを持っています。


|比較項目|Hugging Face (`datasets.Dataset`)|PyTorch (`torch.utils.data.Dataset`)|
|---|---|---|
|主な役割|ディスク上にある大量のテキストや画像を、メモリを節約しながら高速に保持・加工（トークナイズ等）する|機械学習のトレーニングループ（DataLoader）が、行列（Tensor）を1サンプルずつ取り出すための共通インターフェース|
|データの状態|主にテキスト（文字列）や辞書、生の画像データなど。|最終的にGPUに送り込める形になった数値テンソル（Tensor）|

実際の学習はPyTorchの仕組みに従い行われることが多いですが、以下の手順で実行できます。

1. Hugging Faceで準備： `load_dataset`でデータをロードし、`.map(tokenizer)`で一括トークナイズする。
2. PyTorch形式へ変換： `.with_format("torch")`というメソッドを呼び出す。これにより、`datasets.Dataset`の中身が自動的に「PyTorchの Dataset としても振る舞えるハイブリッドな状態」にカプセル化されます。
3. DataLoaderへ投入： 2で変形したデータセットを、そのまま PyTorchの`DataLoader`に放り込んでミニバッチを作成し、GPUへ転送して学習を回します。

個々のデータを見てみます。

In [ ]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

前章で扱ったテキスト分類では"input_ids"（および"attention_mask"等の追加情報も含める）にトークン化した入力データを、"labels"に正解データを含めたバッチデータをモデルに渡していましたが、今回のParaphraseでは入力データを"sentence1"と"sentence2"に分けています。このようにタスクに応じてモデルに渡すべき適切なキー名を把握しておくことが重要です。

このデータセットでは、テキストは`datasets.arrow_dataset.Column`、正解ラベルは`ClassLabel`という特殊なクラスで格納されています。元々はテキストはlist形式で出力されていたので直接Tokenaizerに渡せましたが、v5以降では特殊クラスになったため、listに変換してからTokenizerに渡す必要があります。

In [ ]:
raw_train_dataset.features
raw_datasets["train"]["sentence1"]

今回は2つの文をワンセットとして扱うため、Tokenizerの第一引数と第二引数に"sentence1"と"sentence2"を渡します。2つの文の間に[SEP]トークンが挿入されていれば成功です

In [ ]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenized_dataset = tokenizer(
    list(raw_datasets["train"]["sentence1"]),
    list(raw_datasets["train"]["sentence2"]),
    padding=True,
    truncation=True,
)
print(tokenizer.decode(tokenized_dataset["input_ids"][0]))

一方で上記方法はデータセット全体に一括でトークナイズを実行するため、トークン化中にデータセット全体を保存できるだけの十分なRAMがある場合にのみ機能します。

実用的には、Apache Arrowの動的読込の仕組みをうまく利用するため、以下のように`map`メソッドを使用し、トークナイザを関数として渡す方法が有用です。`map`メソッドの引数`batched=True`を渡すことで、関数がデータセットの複数の要素に一括適用され、前処理が高速化されます。この際、トークナイザに`padding=True`を指定すると、二重でパディングが行われて非効率となるため、指定しないようにしてください

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

学習で必要なのは"input_ids"列なので

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names
tokenized_datasets

また、TokenizerでのPaddingはそのままではデータセット全体で固定長になってしまいますが、これは最大シーケンス長が短いバッチでは無駄なパディングにより処理が遅くなる原因となります。
よってバッチごとに最大シーケンス長を動的にパディング最大長に設定できる`transformers.DataCollat​​orWithPadding`を利用すると、特に学習のデータ読込を高速化することができます。

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 学習

前処理の項で述べたように、以下の方法でデータとトークナイザ、collatorを準備し、モデルのインスタンスも作成しておきます。

- `datasets.load_dataset`関数によるデータセット読込
- トークナイザーを`map(batched=True)`で関数化、Apache Arrowの動的読込の仕組みを利用できるようにする
- `transformers.DataCollatorWithPadding`で動的パディングするcollatorを作成

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification

# Dataset
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")

# Tokenizer
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

学習には以下の2つのクラスから設定とTrainerを作成して利用します

- `transformers.TrainingArguments`クラス: 学習率やバッチサイズ、訓練エポック数、GPU数、混合精度学習等のパラメータを保持
- `transformers.Trainer`クラス: モデルやデータセット、設定を渡して実際に学習を実行するクラス

`transformers.TrainingArguments`クラスで設定できる内容については4章で改めて解説しますが、第一引数`output_dir`に出力先フォルダを指定する必要があります。

In [ ]:
from transformers import TrainingArguments
from transformers import Trainer

training_args = TrainingArguments("/workspace/models/trainers/test-trainer")

以下の例では`eval_strategy`引数で評価の実行タイミングを、`fp16`引数で混合精度学習を指定しています。

In [ ]:
training_args = TrainingArguments(
    "/workspace/models/trainers/test-trainer",
    eval_strategy="epoch",
    fp16=True,  # Enable mixed precision
)

`transformers.Trainer`クラスには、各引数に以下オブジェクトを渡します

- 第1引数: 上で作成したモデルのインスタンス
- 第2引数: `transformers.TrainingArguments`クラスから作成した設定
- `train_dataset`引数: データセットのうち"train"サブセット
- `eval_dataset`引数: データセットのうち"validation"サブセット
- `data_collator`引数: `transformers.DataCollatorWithPadding`クラスから作成したcollator
- `processing_class`引数: 上で作成したTokenizer
- `compute_metrics`引数: 学習中の性能評価用の関数（下の例には含んでいないので後述）

In [ ]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

`train`メソッドで学習を実行します。

In [ ]:
trainer.train()

## 推論

推論は、trainerの`predict`メソッドで実行できます。出力のメンバ変数`predictions`でロジットを取得できます。

In [ ]:
import numpy as np

predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)
print(predictions.predictions[0])

出力のメンバ変数`label_ids`で推論したラベルを取得できます。これはロジットにargmax関数を適用したものと同じです。またロジットにソフトマックス関数を適用するとクラス確率が計算できます。

In [ ]:
import torch

preds = np.argmax(predictions.predictions, axis=-1)
print(preds[0], predictions.label_ids[0])

probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1)
print(probs[0])

## 性能評価

性能評価には`evaluate`ライブラリを使用します。`evaluate.load`メソッドにデータセットと同様の引数（データセット名とサブデータセット名）を指定すると、データセットに準備された性能評価手法が読み込まれます。

`compute`メソッドで、`predictions`引数に推論結果を、`references`引数に正解ラベルを指定すると、性能指標（今回のケースではaccuracyとf1）が計算されます。

In [ ]:
import evaluate

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

実用上は、学習しながら性能評価指標を計算し、学習の進展具合を確認したいケースが多いでしょう。

このケースでは、Trainerから得られるvalidationデータの推論結果を受け取って性能指標を返す関数（この処理はモデルによっても異なるため、ドキュメント確認やデバッグをしながら実装していきます）を作成した上で、Trainerの`compute_metrics`メソッドに渡します、

In [ ]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments("test-trainer", eval_strategy="epoch")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()